# Native swap workflow
Native local quantum program, synthesis, CPU execution and named readout. Seeded host samples are drawn from exact local probabilities; no remote sampling claim. Run every cell from the repository root; see WORKFLOW.md for scope and adaptations. Retain the original 100000 shots. Dropped environment remains entangled and is not clean workspace. Use the exact original seed0 NumPy amplitudes, rather than rounded Qmod constants. Replace the source assertion failure message that refers to an undefined RTOL with an absolute-tolerance message.


In [ ]:
import math
from collections import Counter
from random import Random

import numpy as np
import flagquantum as fq
from examples.synthesis.algorithms.quantum_primitives.swap_test.main import main
from flagquantum import synth

# RandomState retains the source's np.random.seed(0) legacy sequence.
rng = np.random.RandomState(0)
amps1 = 1 - 2 * rng.rand(8)
amps2 = 1 - 2 * rng.rand(8)
amps1 = amps1 / np.linalg.norm(amps1)
amps2 = amps2 / np.linalg.norm(amps2)


In [ ]:
program = synth.synthesize(main, classical_inputs={"first": amps1.tolist(), "second": amps2.tolist()}, max_workspace=0)
result = fq.run(program)
probabilities = dict(program.probabilities(result, registers=("test",)))
NUM_SHOTS = 100_000
counts = Counter(Random(17).choices(list(probabilities), weights=list(probabilities.values()), k=NUM_SHOTS))
overlap_from_swap_test = math.sqrt(2 * counts[(0,)] / NUM_SHOTS - 1)
exact_overlap = float(np.abs(amps1 @ amps2))
print("Sampled overlap:", overlap_from_swap_test)
print("Exact overlap:", exact_overlap)
ATOL = 0.05
assert abs(overlap_from_swap_test - exact_overlap) < ATOL, "Sampled overlap exceeds the absolute tolerance."
print(program.report["resources"])
